# Sentinel AI — Road Damage Detection Model Training
### Trains YOLO11m / YOLOv8m on the RDD2022 (Road Damage Dataset)

This notebook trains a **Medium-tier YOLO detector** for **100 epochs** with **high-resolution feature extraction (`imgsz=1280`)** to ensure high-confidence detection on fine cracks and potholes.

> **GPU Setup**: In Google Colab, go to `Runtime` -> `Change runtime type` -> select `T4 GPU`.

In [ ]:
# Step 1: Check GPU environment and install Ultralytics
!nvidia-smi
!pip install -q ultralytics roboflow

### Step 2: Download Dataset (RDD2022 / Roboflow)
You can download the curated RDD2022 road damage dataset in YOLO format via Roboflow:

In [ ]:
from roboflow import Roboflow

# Download the benchmark Road Damage Dataset (RDD2022) pre-formatted for YOLO
# If using your own Roboflow API key, replace below:
rf = Roboflow(api_key="YOUR_ROBOFLOW_API_KEY")
project = rf.workspace("crowdsensing").project("rdd2022")
dataset = project.version(1).download("yolov8")

print("Dataset downloaded to:", dataset.location)

### Step 3: Verify Classes in `data.yaml`
Sentinel AI expects the standard 4 classes:
- `0: Longitudinal Crack`
- `1: Transverse Crack`
- `2: Alligator Crack`
- `3: Pothole`

In [ ]:
import yaml
from pathlib import Path

yaml_path = Path(dataset.location) / "data.yaml"
with open(yaml_path, 'r') as f:
    data_config = yaml.safe_load(f)

print("Dataset configuration:")
print(yaml.dump(data_config, sort_keys=False))

### Step 4: Train High-Confidence YOLO11m Model
Key hyperparameters tuned specifically to achieve **high confidence scores (80%–95%)** on road distress:
- **`model='yolo11m.pt'`**: 20M parameters (8x the capacity of YOLOv8 Nano).
- **`epochs=100`**: Drives classification loss down so output probabilities approach 1.0.
- **`imgsz=1280`**: Resolves fine 2–5px thin cracks that blur out at 640px.
- **`cls=1.5`**: Penalizes classification errors more heavily.
- **`cos_lr=True`**: Smooth cosine annealing learning rate scheduler.

In [ ]:
from ultralytics import YOLO

# Load pretrained YOLO11 Medium weights (or 'yolov8m.pt')
model = YOLO('yolo11m.pt')

results = model.train(
    data=str(yaml_path),
    epochs=100,
    patience=20,
    batch=16,
    imgsz=1280,
    device=0,
    optimizer='AdamW',
    lr0=0.001,
    lrf=0.01,
    cos_lr=True,
    cls=1.5,
    mosaic=1.0,
    mixup=0.1,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    fliplr=0.5,
    project='sentinel_training',
    name='yolo11m_road_damage',
    save=True,
    plots=True,
    verbose=True
)

print("Training complete!")

### Step 5: Validate and Inspect Results

In [ ]:
# Validate best weights
best_model = YOLO('sentinel_training/yolo11m_road_damage/weights/best.pt')
metrics = best_model.val(data=str(yaml_path))

print("=" * 50)
print(f"Validation mAP@50    : {metrics.box.map50 * 100:.2f}%")
print(f"Validation mAP@50-95 : {metrics.box.map * 100:.2f}%")
print("=" * 50)

### Step 6: Download `best.pt`
Download the generated weights file to replace `backend/ai/models/best.pt` in your Sentinel AI repository.

In [ ]:
from google.colab import files
files.download('sentinel_training/yolo11m_road_damage/weights/best.pt')